In [ ]:
#General imports
import sys, os
import math
import numpy as np
import pandas as pd
import time
from IPython.display import display
import matplotlib.pyplot as plt

#Astropy imports
import astropy.cosmology as Cosmology
from astropy import units as u 
from astropy import constants as const
from astropy.coordinates import SkyCoord

#Own-code import
from repo_root import *
from helper_functions.analytic_neutrino_flux import *
from helper_functions.gamma_flux import Flux_gamma  # NEW: gamma-ray extension
from helper_functions.imf_calibrations import SNr_IMF, LSUN_ERG_S  # NEW: shared calibration


### Used GOALS data

## Herschel IR luminosity data

In [ ]:
file_path = os.path.join(repo_root, "data", "split-lir", "goals_herschel_sample_list.txt")
Herschel = np.loadtxt(file_path)

Split LIR values: https://goals.ipac.caltech.edu/data_files/Lir_LirSurfaceDensity.dat -> IDs missing: 45, 46 , 78, 83, 110, 150, 157,  159 ,187, and 189 + 92 & 238 NaN values. This results in LIR values for 229 galaxies. Note the GOALS sample consists of 202 objects. See also the paper related to the analysis: https://ui.adsabs.harvard.edu/abs/2017ApJ...846...32D/abstract
 
Names corresponding to the iDS in the above file: https://goals.ipac.caltech.edu/data_files/goals_herschel_pacs_CII158_linecatalog_HIPEv13.dat

The redshift and luminosity distance values can be found in "/Users/yarno/documents/PhD/GOALS/goals_herschel_sample_list.txt"

In [ ]:
file_path = os.path.join(repo_root, "data", "split-lir", "lir_split.txt")
LIR_split_array = np.loadtxt(file_path, skiprows=0)
LIR_split = [i[1] for i in LIR_split_array]
LIR_split_unc = [i[2] for i in LIR_split_array]
ID = [int(i[0]) for i in LIR_split_array]

LIR_irg = [i[1] for i in LIR_split_array if i[1] < 1]
LIR_lirg = [i[1] for i in LIR_split_array if i[1] >= 1 and i[1] < 10]
LIR_ulirg = [i[1] for i in LIR_split_array if i[1] >= 10]

SigmaIR = [i[3] for i in LIR_split_array ]
SigmaIR_noNaN = []
for i in SigmaIR:
    if math.isnan(i) == False:
        SigmaIR_noNaN += [i]
    else: 
        SigmaIR_noNaN += [0]



## Individual  $\langle \alpha_{\mathrm{AGN}} \rangle$ distribution

In [ ]:
fig3 = plt.figure(figsize=(10,9))
file_path = os.path.join(repo_root, "data", "split-lir", "agn_split.txt")
AGN_split_array = np.loadtxt(file_path, skiprows=0)
agn_fracs_unc = [i[1] for i in AGN_split_array]
agn_fracs = [i[0] for i in AGN_split_array]
median_ang_frac = np.median(agn_fracs)


agn_fracs = np.array(agn_fracs)


The AGN values can be found in Table 2 of this article: https://iopscience.iop.org/article/10.3847/1538-4357/aa81d7#apjaa81d7t1 . It is noted that the IDS 45, 46 , 78, 83, 110, 150, 157,  159 ,187 and 189 are already missing and therefore 92 & 238 had to be removed.


Note, that the mid-infrared and bolometric AGN fractions are both derived from the Spitzer low-res spectra, and are therefore representative of the projected physical area covered by the IRS short-low slit, centered on the nucleus. For more distant or point-like GOALS sources (D > ~ 50-100 Mpc), the AGN fractions will be representative of the values for the entire galaxy.  However, for more nearby sources, the true global mid-infrared and bolometric AGN fractions can be significantly smaller than those reported here. For example, the source with an average AGN fraction of one is NGC 1068, the most nearby (~16 Mpc) Seyfert II galaxy. The true global bolometric AGN fraction will therefore, most likely, be lower.

# Constructing a general dataframe

In [ ]:
def SNr(LIR): # SN-LIR empirical callibration S. Matilla
    return 2.7e-12*LIR

def SFR(LIR,calib):
    if calib == "Murphy":
        return 3.15e-44*LIR*1e7*3.828e26
    if calib == "Yarno NK":
        return 4.934702e-44*LIR*1e7*3.828e26 
    if calib == "Yarno TH":
        return 1.537522e-44*LIR*1e7*3.828e26

Important note: The commented calibration factor under "Yarno NK" assumes a non-standard SB99 simulation parameter. Specifically, the commented calibration factor takes into account that black holes are already formed from 40 stellar masses. The one that is used for further calculations takes the standard SB99 parameters.

The callibration for the SFR is based on 33GHz continuum data for 56 nuclei and 62 extranuclear regions for star-forming galaxies covering a wide range of integrated properties, ISM conditions, morphological types, IR luminosity range, and and star-formation rates. The Median value for the ratio between the 33GHz star-formation rate and the IR luminosity are consistent for nuclear and extra-nuclear regions. This empirically determined coefficient is consistent with a theoretical relation given in Murpy et al (2011, SFR = 3.88e-44L$_{IR}$,https://ui.adsabs.harvard.edu/abs/2011ApJ...737...67M/abstract and https://ui.adsabs.harvard.edu/abs/2012ApJ...761...97M/abstract).

In [ ]:
file_path = os.path.join(repo_root, "data", "split-lir", "ids.txt")
a_file = open(file_path, "r")

list_of_lists = [] 
for line in a_file:
    stripped_line = line.strip()
    line_list = stripped_line.split()
    list_of_lists.append(line_list)
missing = [45, 46 , 78, 83,92, 110, 150, 157,  159 ,187, 189,238]

l = [[int(i[0]),i[1],i[6],i[7]] for i in list_of_lists if int(i[0]) not in missing]
l2 = [[i[3],i[4]] for i in Herschel if int(i[0]) not in missing]

dec_l_rad = np.array([i[2] for i in Herschel if int(i[0]) not in missing ])*0.0174532925

chars = [
    
          [
        
          ID[i],
          
          round(np.log10(LIR_split[i]*1e11),2), 
          
          round(LIR_split_unc[i],2), 
          
          agn_fracs[i], 
          
          agn_fracs_unc[i], 
          
          round(SFR((1-agn_fracs[i])*LIR_split[i]*1e11,"Yarno NK"),2),
                    
          round(SFR(LIR_split[i]*1e11,"Yarno NK"),2) ,
              
          SigmaIR[i]
          
         ]
         
          for i in range(len(ID)) if ID[i] != 92 and ID[i] != 238 
         ]

    
for i in range(len(chars)): 
    chars[i].insert(1,l[i][1])
    chars[i].insert(2,l[i][2])
    chars[i].insert(3,l[i][3])
    chars[i].insert(4,l2[i][0])
    chars[i].insert(5,l2[i][1])


(!) It should be noted that the neutrino flux predictions already take into account that G=0.5.

In [ ]:
E = 1e3
nism = 1000 
R = 250
v= 500
pmax = 1e8
H = 150
G= 0.5
df = pd.DataFrame({ 'Name' : [i[1] for i in l],
        
        'RA' : [i[2] for i in l ],
        
        'Dec' : [i[3] for i in l],
                   
        'Redshift': [i[4] for i in chars],
        
        'D_L [Mpc]' : [ i[5] for i in chars],
        
        'log(LIR)' : [i[6] for i in chars],
        
        'LIR_unc x 1e11' : [i[7] for i in chars],
        
        'AGNbol' : [i[8] for i in chars],
        
        'AGNbol_unc' : [i[9] for i in chars],
        
        'SFR [M$_{\odot}$]' : [i[10] for i in chars ],
                           
        'un-corr SFR [M$_{\odot}$]':[i[11] for i in chars ] ,
                   
        'Supernova rate [yr$^{-1}$] ' : [round(G*SNr_IMF((1-i[8])*pow(10,i[6]), "Yarno NK"),2) for i in chars],
                   
        'un-corr Supernova rate [yr$^{-1}$] ' : [round(G*SNr_IMF(pow(10,i[6]), "Yarno NK"),2) for i in chars],
        
        'Flux(TeV) [GeV cm$^{-2}$ s$^{-1}$]': [round(Flux(1e3,R,v,nism,H,4,pmax,G*SNr_IMF((1-i[8])*pow(10,i[6]), "Yarno NK"),i[5]),14) for i in chars],
                                                                 
        'Flux(TeV) no AGN [GeV cm$^{-2}$ s$^{-1}$]': [round(Flux(1e3,R,v,nism,H,4,pmax,G*SNr_IMF(pow(10,i[6]), "Yarno NK"),i[5]),14) for i in chars],                   
        
                    
       }, index = [i[0] for i in chars])

pd.set_option('display.max_rows', 500)

display(df)


In [ ]:
# gamma-ray flux columns (mirrors the neutrino Flux(TeV) columns above) ---
# Uses the same fixed source geometry (R, v, nism, H, gammasn, pmax) as the
# neutrino columns, and internal gamma-gamma absorption on each source's own
# FIR photon field (estimated from its own log(LIR), same as the AGN-corrected
# / un-corrected split used for the Supernova rate columns). No EBL absorption
# is applied here (internal_abs only) -- see GOALS_gamma_per_source_predictions.ipynb
# / GOALS_gamma_diffuse_predictions.ipynb for EBL-inclusive versions and for
# energies other than 1 TeV.

#T_dust_default = 40.0  # K -- shared default dust temperature for the whole
#                        # sample (same spirit as the shared gammasn=4 already
#                        # used above: GOALS gives no per-source dust
#                        # temperature any more than it gives a per-source
#                        # spectral index)
#
#gamma_flux_agn_corr, gamma_flux_no_agn = [], []
#for i in chars:
#    RSN_corr = round(G * SNr_IMF((1 - i[8]) * pow(10, i[6]), "Yarno NK"), 2)
#    RSN_no_agn = round(G * SNr_IMF(pow(10, i[6]), "Yarno NK"), 2)
#    L_IR_corr_erg_s = (1 - i[8]) * pow(10, i[6]) * LSUN_ERG_S
#    L_IR_tot_erg_s = pow(10, i[6]) * LSUN_ERG_S
#
#    fg1 = Flux_gamma(E, R, v, nism, H, 4, pmax, RSN_corr, i[5],
#                      internal_abs=True, L_IR=L_IR_corr_erg_s,
#                      T_dust=T_dust_default) if RSN_corr > 0 else 0.0
#    fg2 = Flux_gamma(E, R, v, nism, H, 4, pmax, RSN_no_agn, i[5],
#                      internal_abs=True, L_IR=L_IR_tot_erg_s,
#                      T_dust=T_dust_default) if RSN_no_agn > 0 else 0.0
#    gamma_flux_agn_corr.append(round(float(fg1), 14))
#    gamma_flux_no_agn.append(round(float(fg2), 14))
#
#df['Gamma Flux(TeV) [GeV cm$^{-2}$ s$^{-1}$]'] = gamma_flux_agn_corr
#df['Gamma Flux(TeV) no AGN [GeV cm$^{-2}$ s$^{-1}$]'] = gamma_flux_no_agn
#
#display(df)


In [ ]:
df.to_csv(os.path.join(repo_root, "data", "tables","dataframe"))  